In [ ]:
#install required liibraries and frameworks
from google.colab import drive
!pip install pypdf
!pip install langchain
!pip install -U langchain-community
#!pip installlangchain-pinecone
# import sentence-transformers
#import langchain
# import flask
#import pypdf
#import python-dotenv
#import pinecone[grpc]
#import langchain-pinecone
# import langchain_community
# import langchain_openai
# import langchain_experimental


In [ ]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

#mount drive
drive.mount("/content/drive")

#extract the data from the pdf file

def load_data(filepath):
    loader = DirectoryLoader(filepath, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

filepath = "/content/drive/MyDrive/AI_CHATBOT/"
documents = load_data(filepath)

print(f"Number of documents loaded: {len(documents)}")

In [ ]:
#split extracted text into chunks
def text_split(documents):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(documents)
    return text_chunks

text_chunks=text_split(documents)
print("length of text chunks", len(text_chunks))
text_chunks

In [ ]:
#create the vector embeddings of chunked data
from langchain.embeddings import HuggingFaceEmbeddings
def hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

embeddings=hugging_face_embeddings()

In [ ]:
#testing the downloaded hugging face model to see if it works
query_result = embeddings.embed_query("how are you")
print("Length of query", len(query_result))



In [ ]:
#creating the vector database in pinecone for huggingface embeddings
!pip install pinecone
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="pinecone_api_key")

index_name = "medchatbot"

pc.create_index(
    name=index_name,
    dimension=384, # Replace with your model dimensions
    metric="cosine", # Replace with your model metric
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

In [ ]:
#converting data chunks into vector embeddings, storing the embedded vectors of the data chunks in pinecone vectordb(upserting embeddings into the pinecode index )

# Replace 'YOUR_ACTUAL_API_KEY' with your Pinecone API key
PINE_CONE_API_KEY = "pinecone_api_key"
import os
os.environ["PINECONE_API_KEY"] = PINE_CONE_API_KEY

#PINE_CONE_API_KEY=os.environ.get("PINE_CONE_API_KEY")


!pip install langchain-pinecone
from langchain_pinecone import PineconeVectorStore
index_name = "medchatbot"
#perform the semantic search opeeration when a query is asked ie. it will look for answer
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,  # Text chunks extracted from the PDF
    index_name=index_name,  # Use the same index
    embedding=embeddings,  # Pass the embeddings instance

)


In [ ]:
#connecting openai as a llm to the vector db as knowledge base
#load the existing vector db(ie. index_name) and pass the associated existing embedding model ie (embeddings) to it
import os
PINE_CONE_API_KEY = "pinecone_api_key"
os.environ["PINECONE_API_KEY"] = PINE_CONE_API_KEY
OPENAI_API_KEY = "openai_key"
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

index_name = "medchatbot"
!pip install langchain-pinecone
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

#retrieve relevant answer (ie. Similarity source operation, i think in terms of the vectors closest to the answers. it will rank their similarity) from index
#the arguments passed to this defines that it should provide 3 sentences as answers
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

#testing model's retrieval ability
retrieved_docs = retriever.invoke("")
#print(retrieved_docs)

#if the above code works good, but the idea is to have the ais responses in a conversational flow, to do this:
#introduce the llm. this will process a users query and give a well put together response
!pip install openai
!pip install langchain-openai
from langchain_openai import OpenAI
llm = OpenAI(temperature=0.4, max_tokens=500)






In [ ]:
#if the above code works, good, but the idea is to have the ais responses in a conversational flow, to do this:
#introduce the llm. this will process a users query and give a well put together response
!pip install langchain-openai
from langchain_openai import OpenAI
llm = OpenAI(temperature=0.4, max_tokens=500)
import os
OPENAI_API_KEY = "openai_key"
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

#create the rag application
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

#this is the system prompt, ie to the llm itself telling it what to do
system_prompt = (
    "You are an assistant for question-answering tasks."
    "Use the following pieces of retrieved context to answer the question."
    "If you don't know the answe, say that you don't know and direct the user to call the number 0123456789 for further enquiries."
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

#this is the prompt template. you pass the system's prompt to it and the user's prompt to it also
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

#this explains the rag functionality. retrieving answers in a conversational way such that, it draws from the knowledge base but also from the LLM
question_answer_chain = create_stuff_documents_chain(llm,prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

#response = rag_chain.invoke({"input": "What is acne"})
#print(response["answer"])

**Updated html file 3- updated javascript code to take out try catch error**

In [ ]:
#make folder/directory templates for html file
os.makedirs("templates",exist_ok=True)

#create and save html file, save it as index.html
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.css">
    <link rel="stylesheet" href="/static/style.css">
    <title>MedChatbot</title>
</head>
<body>
    <div class="wrapper">
        <div class="title">MedAssist</div>
        <div id="chat-box" class="box">
            <div class="item">
                <div class="icon">
                    <i class="fa fa-user"></i>
                </div>
                <div class="msg">
                    <p>Hello, How may I assist you?</p>
                </div>
            </div>
            <br clear="both">
        </div>

        <div class="typing-area">
            <div class="input-field">
                <input id="user-input" type="text" placeholder="Type your message" required>
                <button id="send-btn">Send</button>
            </div>
        </div>
    </div>

    <script>
        // Select DOM elements
        const sendButton = document.getElementById("send-btn");
        const userInput = document.getElementById("user-input");
        const chatBox = document.getElementById("chat-box");

        console.log("JavaScript loaded successfully."); // Debug: Confirm script loaded

        // Function to fetch ngrok URL
        async function getNgrokUrl() {
            try {
                const response = await fetch("/get-ngrok-url");
                if (!response.ok) {
                    throw new Error(`Failed to fetch ngrok URL. Status: ${response.status}`);
                }

                const ngrokUrl = await response.text();
                console.log("Ngrok URL fetched:", ngrokUrl); // Debug: Log fetched URL
                return ngrokUrl;
            } catch (error) {
                console.error("Error fetching ngrok URL:", error);
                alert("Unable to connect to the server. Please try again later."); // User-friendly message
                return null;
            }
        }

        // Function to handle sending the message
        async function sendMessage() {
            console.log("Send button clicked."); // Debug: Log button click

            const message = userInput.value.trim();
            if (message === "") {
                console.warn("No message entered."); // Debug: Warn about empty input
                alert("Please enter a message before sending."); // User-friendly message
                return;
            }

            // Display user's message in the chatbox
            const userMessage = document.createElement("div");
            userMessage.className = "item user";
            userMessage.innerHTML = `
                <div class="icon">
                    <i class="fa fa-user"></i>
                </div>
                <div class="msg">
                    <p>${message}</p>
                </div>
            `;
            chatBox.appendChild(userMessage);
            userInput.value = ""; // Clear input field

            try {
                const ngrokUrl = await getNgrokUrl(); // Fetch ngrok URL
                if (!ngrokUrl) {
                    console.warn("Ngrok URL not available."); // Debug: Warn about missing URL
                    return;
                }

                const response = await fetch(`${ngrokUrl}/get`, {
                    method: "POST",
                    headers: {
                        "Content-Type": "application/json",
                    },
                    body: JSON.stringify({ message: message }),
                });

                if (!response.ok) {
                    throw new Error(`Failed to fetch response from server. Status: ${response.status}`);
                }

                const data = await response.text();

                // Display the bot's response
                const botMessage = document.createElement("div");
                botMessage.className = "item bot";
                botMessage.innerHTML = `
                    <div class="icon">
                        <i class="fa fa-robot"></i>
                    </div>
                    <div class="msg">
                        <p>${data}</p>
                    </div>
                `;
                chatBox.appendChild(botMessage);

                // Scroll to the latest message
                chatBox.scrollTop = chatBox.scrollHeight;
            } catch (error) {
                console.error("Error sending message:", error);
                alert("Unable to send your message. Please try again later."); // User-friendly message
            }
        }

        // Add event listener to the button
        sendButton.addEventListener("click", sendMessage);

        // Allow pressing "Enter" to send the message
        userInput.addEventListener("keypress", (event) => {
            if (event.key === "Enter") {
                sendMessage();
            }
        });
    </script>
</body>
</html>
"""
with open("templates/index.html", "w") as html_file:
    html_file.write(html_content)

**Updated html file 2 - updating javascript to prevent displaying errors on front end**

In [ ]:
#make folder/directory templates for html file
os.makedirs("templates",exist_ok=True)

#create and save html file, save it as index.html
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.css">
    <link rel="stylesheet" href="/static/style.css">
    <title>MedChatbot</title>
</head>
<body>
    <div class="wrapper">
        <div class="title">MedAssist</div>
        <div class="box" id="chat-box">
            <div class="item">
                <div class="icon">
                    <i class="fa fa-user"></i>
                </div>
                <div class="msg">
                    <p>Hello, How may I assist you?</p>
                </div>
            </div>
            <br clear="both">

        </div>

        <div class="typing-area">
            <div class="input-field">
                <input type="text" id="user-input" placeholder="Type your message" required>
                <button id="send-btn">Send</button>
            </div>
        </div>
    </div>
<script>
        // Select DOM elements
        const sendButton = document.getElementById("send-btn");
        const userInput = document.getElementById("user-input");
        const chatBox = document.getElementById("chat-box");

        async function getNgrokUrl() {
          try {
              const response = await fetch("/get-ngrok-url");
              const data = await response.json();
              return data.url;
          } catch (error) {
            console.error("Error fetching ngrok URL:", error);
            return null;
          }
        }

        // Function to handle sending the message
        async function sendMessage() {
            const message = userInput.value.trim();
            console.log("User message:", message);
            if (message === "") return; // Prevent empty messages

            // Display user's message
            const userMessage = document.createElement("div");
            userMessage.className = "item user";
            userMessage.innerHTML = `
                <div class="icon">
                    <i class="fa fa-user"></i>
                </div>
                <div class="msg">
                    <p>${message}</p>
                </div>
            `;
            chatBox.appendChild(userMessage);
            userInput.value = ""; // Clear input field

            try {
                // Dynamically fetch ngrok URL
                const ngrokUrl = await getNgrokUrl();
                if (!ngrokUrl) {
                    throw new Error("Failed to retrieve ngrok URL");
                }

                const response = await fetch(`${ngrokUrl}/get`, {
                  method: "POST",
                  headers: {
                    "Content-Type": "application/json",
                  },
                  body: JSON.stringify({ msg: message }),
              });

              const data = await response.text();

            // Send the message to the backend
            try {
                console.log("Sending message to the backend");
                const response = await fetch(`${ngrokUrl}/get`, {
                    method: "POST",
                    headers: {
                        "Content-Type": "application/json",
                    },
                    body: JSON.stringify({ msg: message }),
                });

                if (!response.ok) {
                  throw new Error(`Server error: ${response.status}`);
                }

                const data = await response.text();

                // Display the bot's response
                const botMessage = document.createElement("div");
                botMessage.className = "item bot";
                botMessage.innerHTML = `
                    <div class="icon">
                        <i class="fa fa-robot"></i>
                    </div>
                    <div class="msg">
                        <p>${data}</p>
                    </div>
                `;
                chatBox.appendChild(botMessage);

                // Scroll to the latest message
                chatBox.scrollTop = chatBox.scrollHeight;
            } catch (error) {
                console.error("Error:", error);

                //display friendly error message
                const errorMessage = document.createElement("div");
                errorMessage.className = "item bot";
                errorMessage.innerHTML = `
                  <div class="icon">
                    <i class="fa fa-exclamation-triangle"></i>
                  </div>
                  <div class="msg">
                    <p>Sorry, something went wrong. Please try again later.</p>
                </div>
            `;
            chatBox.appendChild(errorMessage);

            // Scroll to the latest message
            chatBox.scrollTop = chatBox.scrollHeight;
            }
          }
        }

        // Add event listener to the button
        sendButton.addEventListener("click", sendMessage);

        // Allow pressing "Enter" to send the message
        userInput.addEventListener("keypress", (event) => {
            if (event.key === "Enter") {
                sendMessage();
            }
        });
    </script>
</body>
</html>
"""
with open("templates/index.html", "w") as html_file:
    html_file.write(html_content)

**Updated html file - also works but it displays error on the front end**

In [ ]:
#make folder/directory templates for html file
os.makedirs("templates",exist_ok=True)

#create and save html file, save it as index.html
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.css">
    <link rel="stylesheet" href="/static/style.css">
    <title>MedChatbot</title>
</head>
<body>
    <div class="wrapper">
        <div class="title">MedAssist</div>
        <div class="box" id="chat-box">
            <div class="item">
                <div class="icon">
                    <i class="fa fa-user"></i>
                </div>
                <div class="msg">
                    <p>Hello, How may I assist you?</p>
                </div>
            </div>
            <br clear="both">

        </div>

        <div class="typing-area">
            <div class="input-field">
                <input type="text" id="user-input" placeholder="Type your message" required>
                <button id="send-btn">Send</button>
            </div>
        </div>
    </div>
<script>
        // Select DOM elements
        const sendButton = document.getElementById("send-btn");
        const userInput = document.getElementById("user-input");
        const chatBox = document.getElementById("chat-box");

        async function getNgrokUrl() {
        try {
            const response = await fetch("/get-ngrok-url");
            const data = await response.json();
            return data.url;
        } catch (error) {
            console.error("Error fetching ngrok URL:", error);
            return null;
        }
      }

        // Function to handle sending the message
        async function sendMessage() {
            const message = userInput.value.trim();
            console.log("User message:", message);
            if (message === "") return; // Prevent empty messages

            // Display user's message
            const userMessage = document.createElement("div");
            console.log("Displaying user's message");
            userMessage.className = "item user";
            userMessage.innerHTML = `
                <div class="icon">
                    <i class="fa fa-user"></i>
                </div>
                <div class="msg">
                    <p>${message}</p>
                </div>
            `;
            chatBox.appendChild(userMessage);
            userInput.value = ""; // Clear input field

            try {
                // Dynamically fetch ngrok URL
                const ngrokUrl = await getNgrokUrl();
                if (!ngrokUrl) {
                    throw new Error("Failed to retrieve ngrok URL");
                }

                const response = await fetch(`${ngrokUrl}/get`, {
                  method: "POST",
                  headers: {
                    "Content-Type": "application/json",
                  },
                  body: JSON.stringify({ msg: message }),
              });

              const data = await response.text();

            // Send the message to the backend
            try {
                console.log("Sending message to the backend");
                const response = await fetch("/get", {
                    method: "POST",
                    headers: {
                        "Content-Type": "application/json",
                    },
                    body: JSON.stringify({ msg: message }),
                });
                const data = await response.text();

                // Display the bot's response
                const botMessage = document.createElement("div");
                botMessage.className = "item bot";
                botMessage.innerHTML = `
                    <div class="icon">
                        <i class="fa fa-robot"></i>
                    </div>
                    <div class="msg">
                        <p>${data}</p>
                    </div>
                `;
                chatBox.appendChild(botMessage);

                // Scroll to the latest message
                chatBox.scrollTop = chatBox.scrollHeight;
            } catch (error) {
                console.error("Error in sendMessage:", error);

                //Display a friendly error message
                const errorMessage = document.createElement("div");
                errorMessage.className = "item bot";
                errorMessage.innerHTML = `
                  <div class="icon">
                    <i class="fa fa-robot"></i>
                  </div>
                  <div class="msg">
                    <p>Sorry, there was an error processing your request. Please try again later.</p>
                  </div>
                `;
                chatBox.appendChild(errorMessage);

            }
        }

        // Add event listener to the button
        sendButton.addEventListener("click", sendMessage);

        // Allow pressing "Enter" to send the message
        userInput.addEventListener("keypress", (event) => {
            if (event.key === "Enter") {
                sendMessage();
            }
        });
    </script>
</body>
</html>
"""
with open("templates/index.html", "w") as html_file:
    html_file.write(html_content)

**Css file**

In [ ]:
#create folder static for css file. save file as style.css
os.makedirs("static",exist_ok=True)

css_content = """
@import url('https://fonts.googleapis.com/css2?family=Poppins:ital,wght@0,100;0,200;0,300;0,400;0,500;0,600;0,700;0,800;0,900;1,100;1,200;1,300;1,400;1,500;1,600;1,700;1,800;1,900&display=swap');

* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

body {
    font-family: 'Poppins', sans-serif;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 100vh;
}

.wrapper {
    width: 370px;
}

.wrapper .title {
    background: #007bff;
    color: white;
    line-height: 2.5;
    border-radius: 5px 5px 0 0;
    text-align: center;
    font-size: 20px;
}

.wrapper .box {
    border: 1px solid #efefef;
    padding: 10px 15px;
    min-height: 400px;
    max-height: 400px;
}

.wrapper .box .item {
    display: flex;
    float: left;
    margin: 10px 0;
}

.wrapper .box .item .icon {
    background: #007bff;
    color: white;
    width: 40px;
    height: 40px;
    text-align: center;
    line-height: 40px;
    border-radius: 50%;
}

.wrapper .box .item .msg {
    background: #007bff;
    color: white;
    border-radius: 10px;
    width: 150px;
    margin-left: 10px;
}

.wrapper .box .item .msg p {
    padding: 10px;
}

.wrapper .box .item.right {
    float: right;
}

.wrapper .box .item.right .msg {
    background: #efefef;
    color: #333;
}

.wrapper .typing-area {
    width: 100%;
    background: #efefef;
    height: 50px;
    display: flex;
    justify-content: center;
    align-items: center;
    padding: 10px;
}

.wrapper .typing-area .input-field {
    width: 100%;
    position: relative;
}

.wrapper .typing-area .input-field input {
    width: 100%;
    padding: 10px;
    border: 1px solid transparent;
    border-radius: 3px;
    outline: none;
    padding-right: 70px;
    font-family: 'Poppins', sans-serif;
    transition: 0.3s all ease;
}

.wrapper .typing-area .input-field input:focus {
    border-color: #007bff;
}

.wrapper .typing-area .input-field button {
    position: absolute;
    top: 50%;
    right: 10px;
    transform: translateY(-50%);
    background: transparent;
    border: 1px solid #007bff;
    padding: 5px 10px;
    border-radius: 3px;
    color: #007bff;
    outline: none;
    cursor: pointer;
    opacity: 0;
    pointer-events: none;
    transition: 0.3s all ease;
}

.wrapper .typing-area .input-field button:hover {
    background: #007bff;
    color: white;
}

.wrapper .typing-area .input-field input:valid ~ button {
    opacity: 1;
    pointer-events: auto;
}
"""

with open("static/style.css","w") as css_file:
  css_file.write(css_content)

**Libraries for running example.py script**

In [ ]:
#installing needed libraries for script example.py
!pip install flask
!pip install pyngrok

**Hopeful final working code 5**

In [ ]:
code = """
# Import necessary libraries
import os
import threading
import signal
from flask import Flask, request, render_template
from pyngrok import ngrok
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# Initialize Flask
app = Flask(__name__)
port = "8081"

# Global variables
rag_chain = None  # Will hold the RAG chain object
server_thread = None  # Flask server thread

def create_ngrok_tunnel():

    #Sets up an ngrok tunnel for public-facing access to the Flask app.

    public_url = ngrok.connect(port).public_url
    print(f' * ngrok tunnel "{public_url}" -> "http://127.0.0.1:{port}"')
    app.config["BASE_URL"] = public_url

# Flask route to provide ngrok URL to frontend
@app.route("/get-ngrok-url", methods=["GET"])
def get_ngrok_url():
    #Provides the public ngrok URL to the frontend.

    try:
        public_url = app.config.get("BASE_URL")
        if not public_url:
            raise ValueError("Ngrok URL not available.")
        return public_url
    except Exception as e:
        print(f"Error in /get-ngrok-url route: {e}")
        return "Error fetching ngrok URL", 500


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/get", methods=["GET", "POST"])
def chat():

    #Handles user queries and returns responses from the RAG chain.
    try:
        data = request.get_json()  # Parse JSON from the request body
        message = data.get("message")  # Extract 'message' from the JSON payload
        if not message:
            print("Error: No message provided in the request.")
            return "Invalid request. No message provided.", 400

        print(f"User: {message}")
        if not rag_chain:
            raise RuntimeError("RAG chain is not initialized.")

        response = rag_chain.invoke({"input": message})
        print(f"Response: {response['answer']}")
        return str(response["answer"])
    except Exception as e:
        print(f"Error in /get route: {e}")
        return "Sorry, there was an error processing your request.", 500




    #try:
     #   msg = request.json["message"]
    #  print(f"User: {msg}")
     #   if not rag_chain:
      #      raise RuntimeError("RAG chain is not initialized.")
       # response = rag_chain.invoke({"input": msg})
        #print(f"Response: {response['answer']}")
        #return str(response["answer"])
    #except Exception as e:
     #   print(f"Error in /get route: {e}")
      #  return "Sorry, there was an error processing your request."


def start_flask_server():

    #Starts the Flask server.

    app.run(host="localhost", port=8081, debug=True, use_reloader=False)


def shutdown_flask_server():

    #Shuts down the Flask server gracefully.

    global server_thread
    if server_thread and server_thread.is_alive():
        print("Shutting down Flask server...")
        # Send a SIGINT signal to terminate the Flask server
        os.kill(os.getpid(), signal.SIGINT)


def main():

    #Main function to initialize and run the RAG chain, ngrok tunnel, and Flask server.

    global rag_chain, server_thread

    try:
        # Step 1: Set environment variables
        os.environ["PINECONE_API_KEY"] = "pinecone_api_key"
        os.environ["OPENAI_API_KEY"] = "openai_key"

        # Step 2: Initialize embeddings
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        print("Embeddings initialized successfully.")

        # Step 3: Load Pinecone vector store
        index_name = "medchatbot"  # Replace with your actual Pinecone index name
        docsearch = PineconeVectorStore.from_existing_index(
            index_name=index_name,
            embedding=embeddings
        )
        print(f"Pinecone index '{index_name}' loaded successfully.")

        # Step 4: Create a retriever object
        retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})
        print("Retriever initialized successfully.")

        # Step 5: Initialize the OpenAI language model
        llm = OpenAI(temperature=0.4, max_tokens=500)
        print("LLM initialized successfully.")

        # Step 6: Define the system prompt
        system_prompt = (
            "You are an assistant for question-answering tasks. "
            "Use the following pieces of retrieved context to answer the question. "
            "If you don't know the answer, say that you don't know and direct the user to call the number 0123456789 for further enquiries. "
            "Use three sentences maximum and keep the answer concise."
            \n\n
            "{context}"
        )

        # Step 7: Create the prompt template
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", system_prompt),
                ("human", "{input}"),
            ]
        )
        print("Prompt template initialized successfully.")

        # Step 8: Create the RAG chain
        question_answer_chain = create_stuff_documents_chain(llm, prompt)
        rag_chain = create_retrieval_chain(retriever, question_answer_chain)
        print("RAG chain initialized successfully.")

        # Step 9: Start the ngrok tunnel
        create_ngrok_tunnel()
        print("ngrok tunnel created successfully.")

        # Step 10: Start the Flask server in a separate thread
        server_thread = threading.Thread(target=start_flask_server)
        server_thread.start()
        print("Flask server started successfully.")
        server_thread.join()  # Keep the main

    except Exception as e:
        print(f"An error occurred during initialization: {e}")
        shutdown_flask_server()
        print("Flask server shutdown due to an error.")
if __name__ == "__main__":
    main()
"""
with open("example.py","w") as f:
  f.write(code)

**Executing example.py with ngrok authtoken**

In [ ]:
!NGROK_AUTHTOKEN=ngrok_token python3 example.py

In [ ]:
#make folders/directories templates and static for html file and css file respectively
os.makedirs("templates",exist_ok=True)
os.makedirs("static",exist_ok=True)

#create and save html file, save it as index.html
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.css">
    <link rel="stylesheet" href="/static/style.css">
    <title>MedChatbot</title>
</head>
<body>
    <div class="wrapper">
        <div class="title">MedAssist</div>
        <div class="box">
            <div class="item">
                <div class="icon">
                    <i class="fa fa-user"></i>
                </div>
                <div class="msg">
                    <p>Hello, How may I assist you?</p>
                </div>
            </div>
            <br clear="both">

        </div>

        <div class="typing-area">
            <div class="input-field">
                <input type="text" placeholder="Type your message" required>
                <button>Send</button>
            </div>
        </div>
    </div>
</body>
</html>
"""
with open("templates/index.html", "w") as html_file:
    html_file.write(html_content)

#create and save css file, save it as style.css
css_content = """
@import url('https://fonts.googleapis.com/css2?family=Poppins:ital,wght@0,100;0,200;0,300;0,400;0,500;0,600;0,700;0,800;0,900;1,100;1,200;1,300;1,400;1,500;1,600;1,700;1,800;1,900&display=swap');

* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

body {
    font-family: 'Poppins', sans-serif;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 100vh;
}

.wrapper {
    width: 370px;
}

.wrapper .title {
    background: #007bff;
    color: white;
    line-height: 2.5;
    border-radius: 5px 5px 0 0;
    text-align: center;
    font-size: 20px;
}

.wrapper .box {
    border: 1px solid #efefef;
    padding: 10px 15px;
    min-height: 400px;
    max-height: 400px;
}

.wrapper .box .item {
    display: flex;
    float: left;
    margin: 10px 0;
}

.wrapper .box .item .icon {
    background: #007bff;
    color: white;
    width: 40px;
    height: 40px;
    text-align: center;
    line-height: 40px;
    border-radius: 50%;
}

.wrapper .box .item .msg {
    background: #007bff;
    color: white;
    border-radius: 10px;
    width: 150px;
    margin-left: 10px;
}

.wrapper .box .item .msg p {
    padding: 10px;
}

.wrapper .box .item.right {
    float: right;
}

.wrapper .box .item.right .msg {
    background: #efefef;
    color: #333;
}

.wrapper .typing-area {
    width: 100%;
    background: #efefef;
    height: 50px;
    display: flex;
    justify-content: center;
    align-items: center;
    padding: 10px;
}

.wrapper .typing-area .input-field {
    width: 100%;
    position: relative;
}

.wrapper .typing-area .input-field input {
    width: 100%;
    padding: 10px;
    border: 1px solid transparent;
    border-radius: 3px;
    outline: none;
    padding-right: 70px;
    font-family: 'Poppins', sans-serif;
    transition: 0.3s all ease;
}

.wrapper .typing-area .input-field input:focus {
    border-color: #007bff;
}

.wrapper .typing-area .input-field button {
    position: absolute;
    top: 50%;
    right: 10px;
    transform: translateY(-50%);
    background: transparent;
    border: 1px solid #007bff;
    padding: 5px 10px;
    border-radius: 3px;
    color: #007bff;
    outline: none;
    cursor: pointer;
    opacity: 0;
    pointer-events: none;
    transition: 0.3s all ease;
}

.wrapper .typing-area .input-field button:hover {
    background: #007bff;
    color: white;
}

.wrapper .typing-area .input-field input:valid ~ button {
    opacity: 1;
    pointer-events: auto;
}
"""

with open("static/style.css","w") as css_file:
  css_file.write(css_content)